# Submitting Batch Jobs with the qBraid Runtime

The [qBraid-SDK](https://github.com/qBraid/qBraid) can submit a list of circuits as a single batch job. Passing `as_batch=True` to `QbraidDevice.run()` sends the circuits in one API call and returns a single `BatchResult` instead of a separate job per circuit. This was added in [qBraid-SDK v0.12.1](https://github.com/qBraid/qBraid/releases/tag/v0.12.1).

> **Note:** Batch submission requires a device whose profile reports `batch_job_support = True`.

In [ ]:
%%capture

%pip install 'qbraid[qiskit,visualization]'

In [ ]:
from qiskit import QuantumCircuit
from qbraid.runtime import QbraidProvider
from qbraid.visualization import plot_histogram

provider = QbraidProvider()
device = provider.get_device("qbraid:equal1:sim:bell-1")

assert device.profile.batch_job_support is True

## Build the circuits

A batch needs more than one circuit, so we build two with Qiskit. `bell` is a two-qubit circuit and `ghz` is a three-qubit circuit — both are standard entangled states. Calling `measure_all()` adds a measurement on every qubit so the device returns counts.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
ghz.measure_all()

## Submit as a single batch

We pass the two circuits as a list with `as_batch=True`, so they run as one job instead of two separate submissions. `job.result()` returns a `BatchResult`: `result.num_circuits` is how many circuits ran, `result.results` holds each circuit's counts on its own, and `result.data.get_counts()` combines them, which we then plot as a histogram.

In [ ]:
job = device.run([bell, ghz], as_batch=True, shots=100)

result = job.result()  # BatchResult
print(result.num_circuits)  # 2

for i, circuit_result in enumerate(result.results):
    print(f"Circuit {i}: {circuit_result.data.get_counts()}")

batch_counts = result.data.get_counts()

plot_histogram(batch_counts)

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>